# NB2c — Building the Fact Cards

I merge everything I extracted into one **fact card** per human article. This card is the
contract between extraction and generation: it is the *only* thing I pass to the generator
models in NB2d–NB2i.

**Input:** `ha_corpus.parquet` + `extract_algo.parquet` (NB2b, entity-merged).
**Output:** `fact_cards.parquet` — the input to every generation notebook.

## Why the cards are built purely from the algorithmic track

I originally planned a dual-track design: an ALLaM-7B neural summarizer (Track A) alongside
the algorithmic extraction (Track B). I ran ALLaM-7B and hit a hard blocker: its tokenizer
does not decode cleanly — I verified four different decode methods
(`clean_up_tokenization_spaces` True/False, `batch_decode`, `convert_tokens_to_string`) and
all four returned raw SentencePiece markers with broken words and split digits
("2 0 0 7", "يُ شير"). Corrupted entities and numbers are fatal for a fact card whose whole
job is to anchor the AI counterpart to the same names, dates and figures. The model also
ignored the requested output format and produced descriptive prose ("النص يتناول...") rather
than extracted facts.

So I dropped Track A. This is not a loss — it's an improvement:

| Property | Algorithmic only |
|---|---|
| Entity/number integrity | exact, straight from the source |
| Hallucination | impossible (nothing is generated) |
| **Summarizer style leak** | **eliminated** — no LLM ever touches the card |
| Reproducibility | total (same input → same card) |
| Cost | zero GPU |

Dropping ALLaM also removes the summarizer-leak risk I was worried about: since no generative
model produces any part of the card, no model fingerprint can propagate into the whole AI class.

## Card fields (mirror rule)

Every field is only requested from the generator **if it exists in the source article** —
except entities, which are always required because they anchor "same event".

In [1]:
import pandas as pd
import numpy as np
import json
import re

HA_PATH   = '/kaggle/input/notebooks/bahaaqassem/nb0b-select-corpus/ha_corpus.parquet'
ALGO_PATH = '/kaggle/input/notebooks/bahaaqassem/nb2b-post-entity-merge/extract_algo.parquet'   # entity-merged version
OUT_DIR   = '/kaggle/working'

ha   = pd.read_parquet(HA_PATH)
algo = pd.read_parquet(ALGO_PATH)
print('ha:', ha.shape, '| algo:', algo.shape)

df = ha.merge(algo, on='id', how='inner', suffixes=('', '_algo'))
print('merged:', df.shape)
assert len(df) == len(ha), 'row count changed on merge'

ha: (3500, 5) | algo: (3500, 10)
merged: (3500, 14)


## Topic core

In news writing (inverted pyramid) the headline + lead carry the core of the story, so I take
the opening ~25 words of the article body. I first strip the site/section noise that survived
NB1b cleaning (agency names, section labels), and I cut at a natural boundary.

In [2]:
AGENCIES = ['الجزيرة نت','الجزيرة','رويترز','فرانس برس','الأناضول','أ ف ب','قنا',
            'وكالة الأنباء','أسوشيتد برس','الوكالة الفلسطينية','وفا','سبوتنيك',
            'العربية','سكاي نيوز','بي بي سي','الشرق الأوسط','واس','بترا']
SECTIONS = ['أخبار','عربي','دولي','سياسة','اقتصاد','رياضة','ثقافة','منوعات','تقارير','رأي']

def clean_noise(s):
    for a in sorted(AGENCIES, key=len, reverse=True):
        s = s.replace(a, ' ')
    for sec in SECTIONS:
        s = re.sub(r'(?<!\S)' + re.escape(sec) + r'(?!\S)', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def build_topic(article_text, max_words=25):
    s = clean_noise(str(article_text))
    head = ' '.join(s.split()[:max_words])
    for sep in ['. ', '، ']:
        idx = head.rfind(sep)
        if idx > len(head) * 0.5:
            return head[:idx].strip()
    return head.strip()

## Fact points

The TextRank key sentences ARE my fact points — real sentences from the source, so no
hallucination is possible. Two problems I measured and fix here:

- Some key sentences are enormous (up to 685 words). Passing a 685-word "fact point" would be
  handing the generator a whole paragraph to rewrite — exactly what my design avoids.
  I shorten each point to ≤40 words, cutting at a natural boundary (never mid-word).
- The first key sentence often carries page-header noise, so I clean it the same way.

In [3]:
def shorten_fact(sentence, max_words=40):
    words = sentence.split()
    if len(words) <= max_words:
        return sentence.strip()
    window = ' '.join(words[:max_words])
    for sep in ['. ', '، ', ': ']:
        idx = window.rfind(sep)
        if idx > len(window) * 0.5:
            return window[:idx].strip()
    return window.strip()   # word boundary fallback

def _overlap(a, b):
    # word-level overlap ratio between two short texts
    wa, wb = set(a.split()), set(b.split())
    return len(wa & wb) / max(len(wa | wb), 1)

def build_fact_points(key_sentences_json, n_points, topic):
    ks = json.loads(key_sentences_json)
    pts = []
    for s in ks:
        p = shorten_fact(clean_noise(s))
        if len(p.split()) < 5:
            continue
        # the first key sentence often restates the headline -> drop it if it echoes the topic
        if _overlap(p, topic) > 0.6:
            continue
        # drop near-duplicates of points I already kept
        if any(_overlap(p, q) > 0.7 for q in pts):
            continue
        pts.append(p)
    return pts[:n_points]

## Entity selection & prefix stripping (what the generator actually receives)

Two problems I measured on the raw entity lists, both of which would hurt generation:

**1. Too many entities.** Median is 21 but the tail reaches 128. Handing a generator 128 names
forces it to cram them all in — producing an unnaturally dense text ("entity stuffing"), which
is itself an obvious machine tell. So I select the entities that actually matter, combining the
two signals of importance in news writing:
- **first 20 by order of appearance** (inverted pyramid: core entities come early), and
- **top 20 by frequency** in the article,
then union them. This yields ~20 entities (max 35) instead of up to 128.

**2. Attached Arabic prefixes.** 22% of entities carry a clitic ("بالقاهرة", "للجزيرة نت").
Passed verbatim, a generator might write "ذهب إلى بالقاهرة" (broken Arabic), and my acceptance
gate would fail to match "بالقاهرة" against the generator's correct "القاهرة".

I strip these **in code, not in the prompt** — adding a "strip the prefixes first" instruction
would pollute the writing task and can't be verified. My stripping is deliberately
**conservative**: I only remove a prefix when the pattern is unambiguous (a clitic followed by
`ال`, or `لل`). I do NOT strip a bare single letter, because Arabic names legitimately start
with those letters — I verified that an aggressive rule destroys real entities
(بايدن→ايدن, باريس→اريس, برلين→رلين). The single-letter cases (وحماس, لعباس) are left intact and
handled later by fuzzy matching at the acceptance gate.

In [4]:
def strip_prefix_safe(e):
    # Conservative: only unambiguous patterns. A clitic + ال, or لل.
    m = re.match(r'^[وفبك](ال.{2,})$', e)     # بال/وال/فال/كال + X -> ال + X
    if m:
        return m.group(1)
    if e.startswith('لل') and len(e) > 4:      # لل + X -> ال + X
        return 'ال' + e[2:]
    return e                                   # everything else untouched (safe)

def select_entities(entities_dict, article_text, first_n=20, top_n=20):
    flat = []
    for cat in ['persons', 'locations', 'organizations']:
        flat.extend(entities_dict.get(cat, []))
    if not flat:
        return []
    pos    = {e: article_text.find(e) for e in flat}          # order of appearance
    counts = {e: article_text.count(e) for e in flat}         # frequency
    by_position = sorted([e for e in flat if pos[e] >= 0], key=lambda e: pos[e])[:first_n]
    by_freq     = sorted(flat, key=lambda e: counts[e], reverse=True)[:top_n]
    merged = list(dict.fromkeys(by_position + by_freq))       # union, appearance order first
    return [strip_prefix_safe(e) for e in merged]

## Assemble the cards (mirror rule applied)

Each card carries **both**:
- `entities` — the full, unmodified list (for documentation and for the fuzzy acceptance gate),
- `entities_for_prompt` — the selected, prefix-stripped list that the generator actually sees.

Keeping both means the gate can still match against every original form, while the prompt stays
clean and bounded.

In [5]:
cards = []
for _, r in df.iterrows():
    entities = json.loads(r['entities'])
    n_ent = sum(len(v) for v in entities.values())
    topic = build_topic(r['text'])
    ents_prompt = select_entities(entities, str(r['text']))

    card = {
        'pair_id':             r['id'],                   # links the AI counterpart back
        'topic_core':          topic,
        'entities':            json.dumps(entities, ensure_ascii=False),      # full, for the gate
        'entities_for_prompt': json.dumps(ents_prompt, ensure_ascii=False),   # what I send
        'numbers_dates':       r['numbers_dates'],        # mirror rule
        'source_agencies':     r['source_agencies'],      # mirror rule
        'quote_events':        r['quote_events'],         # mirror rule
        'fact_points':         json.dumps(
                                   build_fact_points(r['key_sentences'], int(r['n_fact_points']), topic),
                                   ensure_ascii=False),
        'n_fact_points':       int(r['n_fact_points']),
        'target_words':        int(r['length_words']),    # length matching
        'n_entities':          n_ent,                     # full count
        'n_entities_prompt':   len(ents_prompt),          # what the generator receives
        'has_agencies':        len(json.loads(r['source_agencies'])) > 0,
        'has_numbers':         len(json.loads(r['numbers_dates'])) > 0,
        'has_quotes':          len(json.loads(r['quote_events'])) > 0,
        'extraction_source':   'algorithmic',
    }
    cards.append(card)

cards_df = pd.DataFrame(cards)
print('cards:', cards_df.shape)

cards: (3500, 16)


## Quality checks before I hand these to the generators

A card with no entities can't anchor "same event", so I flag those explicitly.

In [6]:
print('cards with 0 entities:', (cards_df['n_entities'] == 0).sum())
print('cards with empty topic:', (cards_df['topic_core'].str.strip() == '').sum())
print('cards with 0 fact points:',
      cards_df['fact_points'].apply(lambda x: len(json.loads(x)) == 0).sum())

print('\n--- mirror-rule coverage (what the generator will be asked for) ---')
print('has agencies:', cards_df['has_agencies'].sum(), f"({100*cards_df['has_agencies'].mean():.0f}%)")
print('has numbers :', cards_df['has_numbers'].sum(),  f"({100*cards_df['has_numbers'].mean():.0f}%)")
print('has quotes  :', cards_df['has_quotes'].sum(),   f"({100*cards_df['has_quotes'].mean():.0f}%)")

print('\n--- sizes ---')
print('entities/card (full)   : mean %.1f | max %d' % (cards_df['n_entities'].mean(), cards_df['n_entities'].max()))
print('entities sent to prompt: mean %.1f | max %d' % (cards_df['n_entities_prompt'].mean(), cards_df['n_entities_prompt'].max()))
fp_len = cards_df['fact_points'].apply(lambda x: max([len(p.split()) for p in json.loads(x)] or [0]))
print('longest fact point (words): max', fp_len.max())
print('topic words: mean %.0f' % cards_df['topic_core'].apply(lambda t: len(t.split())).mean())
print('target_words: median', int(cards_df['target_words'].median()))

# sanity: prefix stripping must never destroy a real name
print('\n--- prefix-strip sanity ---')
for name in ['بايدن','باريس','برلين','بغداد','بيروت','لبنان','فيسبوك','كتائب','فلسطين']:
    assert strip_prefix_safe(name) == name, f'BROKE {name}'
for a, b in [('بالقاهرة','القاهرة'), ('للجزيرة نت','الجزيرة نت'), ('والجهاد الإسلامي','الجهاد الإسلامي')]:
    assert strip_prefix_safe(a) == b, f'FAILED {a}'
print('all name-preservation and stripping checks passed')

cards with 0 entities: 2
cards with empty topic: 0
cards with 0 fact points: 0

--- mirror-rule coverage (what the generator will be asked for) ---
has agencies: 2961 (85%)
has numbers : 3257 (93%)
has quotes  : 3245 (93%)

--- sizes ---
entities/card (full)   : mean 20.6 | max 107
entities sent to prompt: mean 18.2 | max 36
longest fact point (words): max 40
topic words: mean 23
target_words: median 591

--- prefix-strip sanity ---
all name-preservation and stripping checks passed


In [7]:
# Inspect two complete cards end to end — exactly what the generator will receive
for i in range(2):
    c = cards_df.iloc[i]
    print('=' * 70)
    print('pair_id     :', c['pair_id'], '| target_words:', c['target_words'])
    print('topic_core  :', c['topic_core'])
    print('entities SENT (%d of %d):' % (c['n_entities_prompt'], c['n_entities']))
    print('   ', json.loads(c['entities_for_prompt']))
    print('agencies    :', json.loads(c['source_agencies']))
    print('numbers     :', json.loads(c['numbers_dates'])[:6])
    print('quote_events:', json.loads(c['quote_events'])[:2])
    print('fact_points :')
    for p in json.loads(c['fact_points']):
        print('   -', p[:90])

pair_id     : HA_00000 | target_words: 458
topic_core  : بارنياع قال إنه لا حاجة لأحد في اليورانيوم المخصب لدرجة 60 في المئة- جيتي وقال رئيس" الموساد" دافيد بارنياع: "إيران لن تملك سلاحا نوويا؛ لا
entities SENT (23 of 24):
    ['الموساد', 'دافيد بارنياع', 'إيران', 'إسرائيل', 'طهران', 'يديعوت أحرونوت', 'واشنطن', 'فيينا', 'دافيد برنياع', 'بيني غانتس', 'موشيه يعلون', 'أمريكا', 'نفتالي بينيت', 'الولايات المتحدة', 'أنتوني بلينكن', 'صياء عبد الرضا', 'ايران', 'جورج دبليو بوش', 'الغارديان', 'تركيا', 'رجب طيب أردوغان', 'وفنلندا', 'السويد']
agencies    : ['الشرق الأوسط', 'قنا', 'واس']
numbers     : ['مارس', '60', '3', '1980']
quote_events: ['قال إنه لا حاجة لأحد في اليورانيوم المخصب لدرجة 60 في المئة- جيت', 'وقال رئيس" الموساد" دافيد بارنياع: "إيران لن تملك سلاحا نوويا؛ لا']
fact_points :
   - بدوره، زعم رئيس الأركان ووزير الأمن الأسبق موشيه يعلون، أن "النظام الإيراني أصبح المصدر ال
   - وأضاف: "بدلا من التفكير في المقام الأول في حل عسكري، يجب على إسرائيل أن تظهر استراتيجية من
pair_id     : HA_0

In [8]:
cards_df.to_parquet(f'{OUT_DIR}/fact_cards.parquet', index=False)
print('saved fact_cards.parquet:', cards_df.shape)

saved fact_cards.parquet: (3500, 16)


## Notes

- `pair_id` is carried on every card. Each generated article inherits it as `source_pair_id`,
  and NB3 keeps both members of a pair in the same train/val/test fold — this is what prevents
  pair leakage (the model learning the topic instead of the style).
- `target_words` drives length matching (±15%) in the generation notebooks, so the two classes
  end up with identical length distributions and length can't become a cheap signal.
- The mirror-rule flags (`has_agencies`, `has_numbers`, `has_quotes`) tell each generation
  notebook which conditional requirements to enforce, and which to leave out entirely.
- `extraction_source = 'algorithmic'` documents that no generative model contributed to any
  card, so no summarizer fingerprint can leak into the AI class.
- Upload as `aigt-fact-cards` for NB2d–NB2i.